SETUP

In [4]:
import pandas as pd

run_id_main = 49095
run_id_comp = 46654
file_main = f"run/{run_id_main}/results_{run_id_main}.tsv"
file_comp = f"run/{run_id_comp}/results_{run_id_comp}.tsv"

In [5]:
# main dataframe
df = pd.read_csv(file_main, sep='\t', header=0)
df.head()

,ID,max_iters,hash
0,00063d88244921d6ec46aeab6866a8e2,6,0c03828ef4c69e47
1,00063d88244921d6ec46aeab6866a8e2,20,72c161d9ad5752e2
2,00072cf107ae1349c8c59a15c5ce4af1,6,9e072dca530d7a40
3,00072cf107ae1349c8c59a15c5ce4af1,20,9e072dca530d7a40
4,00076733bdbce94d7e44eca84f1425f0,6,845c778944ce7648


In [2]:
# dataframe for comparisons
df_comp = pd.read_csv(file_comp, sep='\t', header=0)
df_comp.head()

,ID,isohash
0,00063d88244921d6ec46aeab6866a8e2,fd5f9925c23132a6988d6b883cabf95d
1,00072cf107ae1349c8c59a15c5ce4af1,55060e6aa0f7a06dc70e163f2296eb95
2,00076733bdbce94d7e44eca84f1425f0,465f956423b4a18f2a53183fe4b5f682
3,000781b7a545fe723159e53127aff659,933f4c028a325d21969a109f37452e78
4,000a41cdca43be89ed62ea3abf2d0b64,fd393fdabc379273b867d42bc2be0ff4


HASH SUMMARY

In [6]:
hash_col = 'hash' if 'hash' in df.columns else 'isohash'

total = len(df)
na = df[hash_col].isna().sum()
unique = df[hash_col].nunique(dropna=True)
same = total - unique - na

print(f"TOTAL: {total}")
print(f"UNIQUE: {unique}")
print(f"NA: {na}")
print(f"SAME: {same}")

if 'max_iters' in df.columns:
    g = df.groupby('max_iters')[hash_col]
    s = pd.DataFrame({
        'TOTAL': g.size(),
        'NA': g.apply(lambda x: x.isna().sum()),
        'UNIQUE': g.nunique(dropna=True)
    })
    s['SAME'] = s['TOTAL'] - s['UNIQUE'] - s['NA']
    print("\nBY max_iters:")
    print(s.to_string())

TOTAL: 62918
UNIQUE: 50557
NA: 63
SAME: 12298

BY max_iters:
           TOTAL  NA  UNIQUE  SAME
max_iters                         
6          31459   9   29739  1711
20         31459  54   29778  1627


COMPARE MAX_ITERS WITH EACH OTHER

GET ALL DUPLICATE HASHES AND THEIR INSTANCES

In [7]:
# adding isohash to duplicated hashes of main to compare
dupe_rows = df[df['hash'].duplicated(keep=False)].sort_values('hash')
merged = dupe_rows.merge(
    df_comp[['ID','isohash']],
    on='ID',
    how='left'
)
print(merged.head(10))

                                 ID  max_iters              hash  \
0  f4b71ea8807414ae20be45f23aacfdec          6  0006a7c83e85548c   
1  f4b71ea8807414ae20be45f23aacfdec         20  0006a7c83e85548c   
2  da1c0781d0932ed263c8b1bb41887ac7          6  000b6569a4652b21   
3  da1c0781d0932ed263c8b1bb41887ac7         20  000b6569a4652b21   
4  a7ada72625357373f1f41dd75bc861e4         20  001aaa48a6b4b4ad   
5  a7ada72625357373f1f41dd75bc861e4          6  001aaa48a6b4b4ad   
6  90351bff2e0fdab91c370abbcbd7273d         20  001d5664e4ea03aa   
7  90351bff2e0fdab91c370abbcbd7273d          6  001d5664e4ea03aa   
8  483f037eb8cd4f395bdf3e78b8b1cd25         20  0020362c6b6260c3   
9  483f037eb8cd4f395bdf3e78b8b1cd25          6  0020362c6b6260c3   

                            isohash  
0  acdb673153485726da9352b99a1f190c  
1  acdb673153485726da9352b99a1f190c  
2  76fb803817f40bdabe5e98bfcb6e3725  
3  76fb803817f40bdabe5e98bfcb6e3725  
4  fd7c7a12449301ed563e3b17e77961d4  
5  fd7c7a12449301ed563e

In [8]:
# adding hash of main to duplicated hashes of isohash
dupe_rows_comp = df_comp[df_comp['isohash'].duplicated(keep=False)].sort_values('isohash')
merged = dupe_rows_comp.merge(
    df[['ID','hash']],
    on='ID',
    how='left'
)
print(merged.head(10))

                                 ID                           isohash  \
0  4666906deb05406c0ecc18de81673d76  00a1a353e84e203de503f206c02b7a2c   
1  4666906deb05406c0ecc18de81673d76  00a1a353e84e203de503f206c02b7a2c   
2  ae07d3502702cd744ad48e254abb192f  00a1a353e84e203de503f206c02b7a2c   
3  ae07d3502702cd744ad48e254abb192f  00a1a353e84e203de503f206c02b7a2c   
4  74acaa627ddc5fdab40d62172f56a827  00c5b384d9311d83da0be53e990b4904   
5  74acaa627ddc5fdab40d62172f56a827  00c5b384d9311d83da0be53e990b4904   
6  66ab346df72c5effa292342037bc3909  00c5b384d9311d83da0be53e990b4904   
7  66ab346df72c5effa292342037bc3909  00c5b384d9311d83da0be53e990b4904   
8  df3da7a967605e804571a36c099416b7  00efc6678bd4710b52c3448cc6e9c5da   
9  df3da7a967605e804571a36c099416b7  00efc6678bd4710b52c3448cc6e9c5da   

               hash  
0  cc59bf95d464f26e  
1  a92e9acdee3d7306  
2  cc59bf95d464f26e  
3  a92e9acdee3d7306  
4  28a3415cc12d2e15  
5  d113037ad7d90966  
6  28a3415cc12d2e15  
7  d113037ad7d90966 